In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datasets
from privacy_estimates.experiments.aml import JobList
from sklearn.metrics import roc_curve, roc_auc_score, auc
import re
import os

In [2]:
def compute_performance(scores_members, scores_non_members):
    mia_performance = {}
    
    member_vals = [val for val in scores_members]
    non_member_vals = [val for val in scores_non_members]
    mia_performance['auc'] = roc_auc_score([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    fpr, tpr, thresholds = roc_curve([1]*len(member_vals) + [0]*len(non_member_vals), member_vals + non_member_vals)
    for target_fpr in (0.01, 0.05, 0.1):
        mia_performance[f'tpr_at_{target_fpr}'] = np.interp(target_fpr, fpr, tpr)
    print(f"AUC: {mia_performance['auc']}, TPR@0.01: {mia_performance['tpr_at_0.01']}, TPR@0.05: {mia_performance['tpr_at_0.05']}, TPR@0.1: {mia_performance['tpr_at_0.1']}")

    # also add the curves
    mia_performance['fpr'] = fpr
    mia_performance['tpr'] = tpr

    return mia_performance

# let's also make a function that computes the performance directly from the url

from datetime import datetime

def compute_performance_from_url(url, job_name = None):
    jobs = JobList.from_urls([url])
    if job_name is None:
        job_name = str(datetime.now())
    
    if not os.path.exists(f'./mia_results/{job_name}'):
        test = jobs[0].get_node('estimate_privacy').download_input('scores', f'./mia_results/{job_name}/scores')
        test = jobs[0].get_node('estimate_privacy').download_input('challenge_bits', f'./mia_results/{job_name}/challenge_bits')
    scores = datasets.load_from_disk(f'./mia_results/{job_name}/scores')
    bits = datasets.load_from_disk(f'./mia_results/{job_name}/challenge_bits')
    
    membership_scores = np.array([k['score'] for k in scores])
    membership_labels = np.array([k['challenge_bit'] for k in bits])
    members = membership_scores[membership_labels == 1]
    non_members = membership_scores[membership_labels == 0]
    return compute_performance(members, non_members)

In [3]:
def extract_aml_urls(log_file_path):
    # Initialize a list to store the extracted URLs
    aml_urls = []

    # Define a regular expression to match the log entries containing AML URLs
    aml_url_pattern = re.compile(r'AML URL: (https://ml\.azure\.com/runs/[\w\-\?&=/]+)')

    # Open and read the log file
    with open(log_file_path, 'r') as file:
        for line in file:
            match = aml_url_pattern.search(line)
            if match:
                aml_urls.append(match.group(1))

    return aml_urls

In [4]:
# first the non synthetic results

perplexities = [10, 10**1.5, 10**2, 10**2.5, 10**3, 10**3.5, 10**4, 10**4.5, 10**5]

no_synthetic_urls = extract_aml_urls('../job_launch_outputs/no_synthetic_ppl_sst2.txt')
synthetic_urls = extract_aml_urls('../job_launch_outputs/synthetic_ppl_sst2.txt')

# do the last one too, which was a relaunch with a small fix
synthetic_url_last_one = extract_aml_urls('../job_launch_outputs/synthetic_ppl_sst2_lastonerelaunch.txt')
synthetic_urls[-1] = synthetic_url_last_one[0]

assert len(no_synthetic_urls) == len(perplexities)
assert len(synthetic_urls) == len(perplexities)


In [ ]:
no_synthetic_urls

In [ ]:
synthetic_urls

In [ ]:
non_synthetic_results = dict()
synthetic_results = dict()

for i in range(len(perplexities)):
    print(f"Perplexity: {perplexities[i]}")
    print("Non synthetic")
    try:
        performance = compute_performance_from_url(no_synthetic_urls[i])
        non_synthetic_results[perplexities[i]] = performance
        print('---')
    except Exception as e:
        print(e)
        non_synthetic_results[perplexities[i]] ={'auc': None, 'tpr_at_0.01': None, 'tpr_at_0.05': None, 'tpr_at_0.1': None}
        print('---')
    
    print('Synthetic')
    try:
        performance = compute_performance_from_url(synthetic_urls[i])
        synthetic_results[perplexities[i]] = performance
        print('---')
    except Exception as e:
        print(e)
        synthetic_results[perplexities[i]] ={'auc': None, 'tpr_at_0.01': None, 'tpr_at_0.05': None, 'tpr_at_0.1': None}
        print('---')

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(perplexities, [non_synthetic_results[p]['auc'] for p in perplexities], '-o', alpha=0.8, 
             color ='darkblue', markersize=10, linewidth=2, markeredgewidth=2, markeredgecolor='white', label=r'Model - $n_{rep}=4$')
plt.plot(perplexities, [synthetic_results[p]['auc'] for p in perplexities], '-o', alpha=0.8, 
             color ='darkorange', markersize=10, linewidth=2, markeredgewidth=2, markeredgecolor='white', label=r'Synthetic (2-gram) - $n_{rep}=16$')

plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label = 'Random guess baseline')


plt.xticks([10**k for k in (0, 1, 2, 3, 4, 5)])
plt.yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0], labels=['0.5', '0.6', '0.7', '0.8', '0.9', '1.0'])

# Enable the grid
plt.grid(True, which="major", ls="--", alpha=0.8)

plt.legend(loc='lower left', fontsize=14)
plt.xlabel('Canary perplexity', fontsize=16)
plt.ylabel('AUC', fontsize=16)
plt.ylim(0.4, 1.02)
plt.xscale('log')
plt.savefig('figures/ppl_experiment_sst2.pdf', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

In [9]:
import pandas as pd
df = pd.DataFrame({'ppl': perplexities, 'model': [non_synthetic_results[p]['auc'] for p in perplexities], 'synthetic': [synthetic_results[p]['auc'] for p in perplexities]})

In [ ]:
from latex import Project
overleaf = Project(url="https://git@git.overleaf.com/667bde737a03ee4008a9359f")
overleaf.push_dataframe(df, 'data/canary_ppl/sst2/auc.tsv')